In [3]:
import pandas as pd
import numpy as np  
from sodapy import Socrata
from datetime import datetime
import os
from tqdm import tqdm
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns


pd.set_option('display.max_columns', None)
pd.set_option("display.max_colwidth", 100)
pd.set_option('display.float_format', '{:.4f}'.format)

# sys.path.append("../src")
sys.path.append(str(Path.cwd().parent))
import utils

path = os.getcwd()
print(path)

path_repo = "C:/Users/PC/Desktop/tesis/"
path_data = "C:/Users/PC/Desktop/tesis/data/"

c:\Users\PC\Desktop\python\rates_degree_work


- La tabla es semanal, terminando en cada viernes

In [4]:
qwe

NameError: name 'qwe' is not defined

## Tasas colocación V1

In [ ]:
tasascol = pd.read_csv(path_repo + "/data/2022.csv", decimal = ",", thousands = ".", encoding='utf-8')
tasascol.columns = [x.lower().replace(" ","_") for x in tasascol.columns]
tasascol.rename(columns = {"tasa_efectiva_promedio_ponderada":"tasa_ponderada", "montos_desembolsados":"monto", "numero_de_creditos_desembolsados":"num_creditos", 
                           "sum_numero_de_creditos":"num_creditos", "producto_de_crédito":"producto", "plazo_de_crédito":"plazo", "nombre_entidad":"banco"}, inplace=True)
tasascol.drop(columns = ["tipo_entidad", "codigo_entidad"], inplace =True)
tasascol.sample(5)

In [ ]:
cols_cat = ['nombre_tipo_entidad', 'banco', 'fecha_corte', 'tipo_de_persona', 'sexo', 'tamaño_de_empresa', 'tipo_de_crédito', 'tipo_de_garantía',
            'producto', 'plazo', 'grupo_etnico', 'antiguedad_de_la_empresa', 'tipo_de_tasa', 'clase_deudor', 'codigo_ciiu', 'codigo_municipio',
            'rango_monto_desembolsado']
cols_cont = ['tasa_ponderada', 'margen_adicional', 'monto', 'num_obl']

tasascol["fecha_corte"] = pd.to_datetime(tasascol["fecha_corte"], dayfirst=True, errors='coerce')

tasascol["tasa_ponderada"] = tasascol["tasa_ponderada"].astype(float)
tasascol["num_creditos"] = tasascol["num_creditos"].astype(int)
tasascol["fecha_corte"] = pd.to_datetime(tasascol["fecha_corte"])
tasascol["anomes"] = tasascol["fecha_corte"].dt.strftime("%Y-%m")

# tasascol = tasascol[~tasascol["banco"].isin(["Fondo Nacional del Ahorro", "La Hipotecaria", "Bancoldex", "Confiar",
#                                         "IRIS C.F.", "Bancamía S.A.", "Bancien", "Mibanco S.A.", "Banco Contactar",
#                                         'Tuya','Crezcamos','Credifamilia','Financiera Juriscoop C.F.','Rappipay',
#                                         'Banco GNB Sudameris','Banco Popular','Coopcentral','Banco Pichincha S.A.','Nu Colombia C.F.',
#                                         "Caja de Vivienda Militar y de Polícia"])] # Excluidos por tasas atipicos
                                        
# tasascol = tasascol[tasascol["tasa_ponderada"] <= 50]
# tasascol = tasascol[tasascol["num_creditos"] >= 2]
# tasascol = tasascol[(tasascol["rango_monto_desembolsado"].notna())] # tienen nulos antes de 202309

tasascol ["banco"] = tasascol["banco"].str.replace("Compañía De Financiamiento","CF").str.replace("Cooperativa Financiera","CF")
tasascol["plazo"] = tasascol["plazo"].replace({'Más de 1 año y hasta 3 años':"(1-3]", 'Más de 3 años y hasta 5 años':"(3-5]",
                                                'Más de 5 años y hasta 7 años':"(5-7]",
                                                'Más de 7 años y hasta 10 años':"(7-10]", 'Más de 10 años':"(10-]",
                                                "Hasta 6 meses":"(0-1]", "Más de 6 meses y hasta 1 año":"(0-1]",
                                                'Hasta 30 días':"(0-1]", 'Entre 31 días y 1 año':"(0-1]"})

tasascol["prom_monto_porobl"] = tasascol["monto"]/tasascol["num_creditos"]

print(tasascol["fecha_corte"].min(), "|", tasascol["fecha_corte"].max())

print(tasascol.shape)
print("Bancos:", tasascol["banco"].nunique())

tasascol.sample(7)

In [ ]:
tasascol[tasascol["anomes"] == "2022-08"]["fecha_corte"].value_counts()

In [ ]:
utils.vc(tasascol["tipo_de_crédito"])

In [ ]:
utils.vc(tasascol["tipo_de_garantía"])

In [ ]:
tasascol.isnull().mean()

In [ ]:
# Sample
tmp = tasascol.drop(columns = ['margen_adicional', 'grupo_etnico', 'antiguedad_de_la_empresa', 'clase_deudor', 'codigo_ciiu',
       'codigo_municipio', 'anomes', 'prom_monto_porobl']).sample(100)
tmp.to_csv(path_data+'sample.csv', index = False)
tmp.sample(5)

In [ ]:
print(np.percentile(tasascol["tasa_ponderada"], 0.05))
print(np.percentile(tasascol["tasa_ponderada"], 2.25))
print(np.percentile(tasascol["tasa_ponderada"], 5))
print(np.percentile(tasascol["tasa_ponderada"], 50))
print(np.percentile(tasascol["tasa_ponderada"], 95))
print(np.percentile(tasascol["tasa_ponderada"], 97.5))
print(np.percentile(tasascol["tasa_ponderada"], 99.5))

In [ ]:
print(tasascol[tasascol["tasa_ponderada"] <= 10].shape)
print(tasascol[tasascol["tasa_ponderada"] >= 40].shape)

In [ ]:
# ax = tasascol[tasascol["fecha_corte"] >= "2025-01-01"]
ax = tasascol
ax = ax.groupby("producto").agg({"monto":"sum", "num_creditos":"sum", "tasa_ponderada":["mean", "median", "min", "max", "std"]}).reset_index()#.sort_values("num_creditos", ascending=False)#.head(30)
ax.columns = ["producto", "monto", "num_creditos", "tasa_mean", "tasa_median", "tasa_min", "tasa_max", "tasa_std"]
ax["tasa_cv"] = ax["tasa_std"]/ax["tasa_mean"].mean()*100
ax["%monto"] = ax["monto"]/ax["monto"].sum()*100
ax["%num_obl"] = ax["num_creditos"]/ax["num_creditos"].sum()*100
ax["prom_monto"] = ax["monto"]/ax["num_creditos"]
ax = ax.sort_values("num_creditos", ascending=False)#.head(30)
ax

In [ ]:
# ax = tasascol[tasascol["fecha_corte"] >= "2025-01-01"]
ax = tasascol
ax = ax.groupby("banco").agg({"monto":"sum", "num_creditos":"sum", "tasa_ponderada":["mean", "median", "min", "max", "std"]}).reset_index()#.sort_values("num_creditos", ascending=False)#.head(30)
ax.columns = ["banco", "monto", "num_creditos", "tasa_mean", "tasa_median", "tasa_min", "tasa_max", "tasa_std"]
ax["tasa_cv"] = ax["tasa_std"]/ax["tasa_mean"].mean()*100
ax["%monto"] = ax["monto"]/ax["monto"].sum()*100
ax["%num_obl"] = ax["num_creditos"]/ax["num_creditos"].sum()*100
ax["prom_monto"] = ax["monto"]/ax["num_creditos"]
ax = ax.sort_values("num_creditos", ascending=False)#.head(30)
ax

In [ ]:
tasascol["anomes"].value_counts(dropna = False, normalize =True)
ax = tasascol
ax = ax.groupby(["anomes"]).agg({"monto":"sum", "num_creditos":"sum", "tasa_ponderada":["mean", "median", "min", "max", "std"]}).reset_index()#.sort_values("num_creditos", ascending=False)#.head(30)
ax.columns = ["mes", "monto", "num_creditos", "tasa_mean", "tasa_median", "tasa_min", "tasa_max", "tasa_std"]
ax["tasa_cv"] = ax["tasa_std"]/ax["tasa_mean"].mean()*100
ax["%monto"] = ax["monto"]/ax["monto"].sum()*100
ax["%num_obl"] = ax["num_creditos"]/ax["num_creditos"].sum()*100
ax["prom_monto"] = ax["monto"]/ax["num_creditos"]
#ax.sort_values("monto", ascending=False).head(40)
ax

In [ ]:
ax = tasascol
ax = ax.groupby(["anomes", "plazo", "rango_desembolso"]).agg({"monto":"sum", "num_creditos":"sum", "tasa_ponderada":"mean"}).reset_index()#.sort_values("num_creditos", ascending=False)#.head(30)
ax["monto"] = ax["monto"]/1000000
ax["prom_monto_porobl"] = ax["monto"]/ax["num_creditos"] # millones

#ax.sort_values("monto", ascending=False).head(40)
tasascol = ax
tasascol

## How to group?

In [ ]:
print(tasascol.shape)
tasascolgr = tasascol.copy()
tasascolgr[cols_cat] = tasascolgr[cols_cat].fillna("NULL")
tasascolgr["tasaXmonto"]  = tasascolgr['tasa_ponderada']*tasascolgr['monto']
tasascolgr = tasascolgr.groupby(['nombre_tipo_entidad', 'banco', 'fecha_corte', 'tipo_de_crédito', 'tipo_de_garantía', 'tipo_de_persona', #'sexo', 'tamaño_de_empresa',  'producto',
                                'plazo', #'grupo_etnico', 'antiguedad_de_la_empresa','tipo_de_tasa', 
                                'rango_monto_desembolsado', #'clase_deudor',
                                #'codigo_ciiu', 'codigo_municipio', 'anomes', 'prom_monto_porobl'
                                ]).agg({"monto":"sum", "tasaXmonto":"sum", 'num_creditos':"sum"}).reset_index()
print(tasascolgr.shape)
tasascolgr.sample(5)

In [ ]:
601806204910.0000/30926219000.0000

In [ ]:
qwwe

## Tasas colocación V2

- Tabla semanal que termina en el viernes de cada semana
- H2002-2026

In [5]:
tasascol = pd.read_csv(path_data+"tasascol_H2002-2026.csv", decimal = ",")
tasascol.columns = [x.lower().replace(" ","_") for x in tasascol.columns.tolist()]
tasascol.rename(columns={'año(aaaa)_-_semana(ss)':"semana", 'fecha_(dd/mm/aaaa)':"fecha", 'nombre_cuenta':"producto", 'clasificación_tipo_establecimiento':"tipo_entidad", 
                         'tasa_(%)':"tasa", 'monto_(miles_de_millones_cop)':"monto"}, inplace =True)
tasascol["fecha"] = pd.to_datetime(tasascol["fecha"], dayfirst = True)
tasascol["anomes"] = tasascol["fecha"].astype(str).str[:7]

print(len(tasascol))
print(min(tasascol["fecha"]))
print(max(tasascol["fecha"]))

tasascol

48961
2002-05-03 00:00:00
2026-10-23 00:00:00


,semana,fecha,producto,tipo_entidad,tasa,monto,anomes
0,2002-18,2002-05-03,Crédito de consumo,Bancos comerciales,28.2454,69367.5781,2002-05
1,2002-18,2002-05-03,Crédito de consumo,Compañías de financiamiento comercial,27.9476,10814.8918,2002-05
2,2002-18,2002-05-03,Crédito de consumo,Cooperativas financieras,27.6983,553.7210,2002-05
3,2002-18,2002-05-03,Crédito de consumo,Organismos cooperativos,29.1372,206.7000,2002-05
4,2002-18,2002-05-03,Crédito de consumo,Total establecimientos,28.2042,80942.8908,2002-05
...,...,...,...,...,...,...,...
48956,2026-42,2026-10-23,Tasa de interés de colocación sin Tesorería,Compañías de financiamiento comercial,20.6723,224703.2947,2026-10
48957,2026-42,2026-10-23,Tasa de interés de colocación sin Tesorería,Cooperativas financieras,20.7742,46398.0847,2026-10
48958,2026-42,2026-10-23,Tasa de interés de colocación sin Tesorería,Corporaciones financieras,16.0700,499.9418,2026-10
48959,2026-42,2026-10-23,Tasa de interés de colocación sin Tesorería,Entidades financieras especiales,17.1214,25851.3850,2026-10


In [ ]:
tasascol.tail(10)

In [ ]:
ax = tasascol[(tasascol["producto"] == "Tasa de interés de colocación Total") & (tasascol["tipo_entidad"] == "Total establecimientos")]
ax["tasaXMonto"] = ax["tasa"] * ax["monto"]
ax = ax.groupby(['anomes', 'producto', 'tipo_entidad'])[['tasaXMonto', 'monto']].sum().reset_index()
ax["tasa"] = ax["tasaXMonto"] / ax["monto"]
del ax["tasaXMonto"]

ax["vari_monto_mes"] = ax.groupby(["producto", "tipo_entidad"])["monto"].pct_change(1)*100
ax["vari_tasa_mes"] = ax.groupby(["producto", "tipo_entidad"])["tasa"].pct_change(1)*100
ax["monto_r12"] = ax.groupby(["producto", "tipo_entidad"])["monto"].shift(12)
ax["tasa_r12"] = ax.groupby(["producto", "tipo_entidad"])["tasa"].shift(12)
ax["y_crec_anual_monto"] = 100*((ax["monto"] / ax["monto_r12"]) - 1)
ax["crec_anual_tasa"] = 100*((ax["tasa"] / ax["tasa_r12"]) - 1)

ax.drop(columns = ["monto_r12", "tasa_r12"], inplace = True)
ax.tail(20)

In [ ]:
tasascol[(tasascol["producto"] == "Tasa de interés de colocación Total") & (tasascol["tipo_entidad"] == "Total establecimientos")].tail(4)

In [ ]:
# Valid ponderacion
display(tasascol[(tasascol["fecha"] == "2026-10-23") & (tasascol["producto"] == "Crédito de consumo") & (tasascol["tipo_entidad"] == "Total establecimientos")])

ax = tasascol[(tasascol["fecha"] == "2026-10-23") & (tasascol["producto"] == "Crédito de consumo")  & (tasascol["tipo_entidad"] != "Total establecimientos")]
ax["tasaXMonto"] = ax["tasa"] * ax["monto"]
print(ax["monto"].sum())
print((ax["tasaXMonto"].sum()/ax["monto"].sum()))
ax

In [ ]:
# Valid ponderacion
display(tasascol[(tasascol["fecha"] == "2026-10-23") & (tasascol["producto"] == "Tasa de interés de colocación Total")])# & (tasascol["tipo_entidad"] == "")])

ax = tasascol[(tasascol["fecha"] == "2026-10-23") & (~tasascol["producto"].isin(["Tasa de interés de colocación Total", "Tasa de interés de colocación Banco de la República Semanal",
                "Tasa de interés de colocación sin Tesorería"])) & (tasascol["tipo_entidad"] != "Total establecimientos")] #& (tasascol["producto"] == "Crédito de consumo")  & (tasascol["tipo_entidad"] != "Total establecimientos")]
ax["tasaXMonto"] = ax["tasa"] * ax["monto"]
ax = ax.groupby(["semana", "fecha", "tipo_entidad"])[["tasaXMonto", "monto"]].sum().reset_index()
ax["tasa"] = ax["tasaXMonto"] / ax["monto"]

print(ax["monto"].sum())
# print((ax["tasaXMonto"].sum()/ax["monto"].sum()))
ax

In [7]:
tasascol2 = tasascol.copy()

## Filters

In [ ]:
tasascol = tasascol2.copy()
tasascol["vari_monto_m1"] = tasascol.groupby(["producto", "tipo_entidad"])["monto"].pct_change(1)*100
tasascol["vari_tasa_m1"] = tasascol.groupby(["producto", "tipo_entidad"])["tasa"].pct_change(1)*100
tasascol["monto_r12"] = tasascol.groupby(["producto", "tipo_entidad"])["monto"].shift(12)
tasascol["tasa_r12"] = tasascol.groupby(["producto", "tipo_entidad"])["tasa"].shift(12)
tasascol["y_vari_anual_monto"] = 100*((tasascol["monto"] / tasascol["monto_r12"]) - 1)
tasascol["vari_anual_tasa"] = 100*((tasascol["tasa"] / tasascol["tasa_r12"]) - 1)

tasascol.drop(columns = ["monto_r12", "tasa_r12", "semana", "tipo_entidad"], inplace = True)

print(len(tasascol))
tasascol = tasascol[(tasascol["tipo_entidad"] == "Bancos comerciales")]
tasascol = tasascol[(~tasascol["producto"].isin(["Tasa de interés de colocación Total", "Tasa de interés de colocación Banco de la República Semanal",
                                                 "Tasa de interés de colocación sin Tesorería"]))]
tasascol = tasascol[(tasascol["fecha"] >= "2003-06-01")]
print(len(tasascol))

print(min(tasascol["fecha"]))

tasascol.tail(15)

KeyError: 'tipo_entidad'

In [ ]:
ax = tasascol[(tasascol["producto"].isin(["Créditos Comerciales (Ordinario)"]))].tail(200)
ax

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 4))
sns.lineplot(data=ax, x="fecha", y="y_vari_anual_monto", color="b", markersize = 5)
plt.title("", fontsize=14, fontweight="bold")
plt.xlabel("", fontsize=12)
plt.ylabel("", fontsize=12)
plt.xticks(rotation=90, fontsize = 8)
plt.axhline(0, color = "gray", linestyle = "--")
plt.tight_layout()
plt.grid(None)
plt.show()

In [ ]:
for i in tasascol["producto"].unique():
    ax = tasascol[tasascol["producto"] == i].tail(200)

    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(15, 4))
    sns.lineplot(data=ax, x="fecha", y="monto", color="b", markersize = 5)  # y_vari_anual_monto
    plt.title(i, fontsize=14, fontweight="bold")
    plt.xlabel("", fontsize=12)
    plt.ylabel("", fontsize=12)
    plt.xticks(rotation=90, fontsize = 8)
    # plt.axhline(0, color = "gray", linestyle = "--")
    plt.tight_layout()
    plt.grid(None)
    plt.show()

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 4))
sns.lineplot(data=ax, x="fecha", y="vari_monto_m1", color="b", markersize = 5)
plt.title("", fontsize=14, fontweight="bold")
plt.xlabel("", fontsize=12)
plt.ylabel("", fontsize=12)
plt.xticks(rotation=90, fontsize = 8)
plt.axhline(0, color = "gray", linestyle = "--")
plt.tight_layout()
plt.grid(None)
plt.show()

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 4))
sns.lineplot(data=ax, x="fecha", y="monto", color="b", markersize = 5)
plt.title("", fontsize=14, fontweight="bold")
plt.xlabel("", fontsize=12)
plt.ylabel("", fontsize=12)
plt.xticks(rotation=90, fontsize = 8)
# plt.axhline(0, color = "gray", linestyle = "--")
plt.tight_layout()
plt.grid(None)
plt.show()

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 4))
sns.lineplot(data=ax, x="fecha", y="tasa", color="b", markersize = 3)
plt.title("", fontsize=14, fontweight="bold")
plt.xlabel("", fontsize=12)
plt.ylabel("", fontsize=12)
plt.xticks(rotation=90, fontsize = 8)
# plt.axhline(0, color = "gray", linestyle = "--")
plt.tight_layout()
plt.grid(None)
plt.show()

In [ ]:
qwe

In [ ]:
# tasascol.tail(200).to_csv(path_data + "tasascol_sample.csv")

## BANREP

In [ ]:
def calc_viernes(df, columna_fecha='Fecha', saltar_fin_de_semana=False):
    df_resultado = df.copy()
    
    # Asegurar formato datetime
    fechas_dt = pd.to_datetime(df_resultado[columna_fecha])
    dia_semana = fechas_dt.dt.weekday
    
    # Calcular días de diferencia hasta el viernes (Viernes = 4)
    if saltar_fin_de_semana:
        dias_al_viernes = np.where(dia_semana > 4, 11 - dia_semana, 4 - dia_semana)
    else:
        dias_al_viernes = 4 - dia_semana
        
    # Asignar la nueva columna
    df_resultado["Fecha_viernes"] = fechas_dt + pd.to_timedelta(dias_al_viernes, unit='d')
    
    return df_resultado


def filtrar_dias_habiles(df, columna_fecha='Fecha'):
    df_resultado = df.copy()
    
    # Asegurar que la columna tenga formato datetime
    fechas_dt = pd.to_datetime(df_resultado[columna_fecha])
    
    # .dt.weekday devuelve 0=Lunes hasta 6=Domingo. 
    # Los días menores a 5 son de lunes a viernes.
    mascara_habiles = fechas_dt.dt.weekday < 5
    
    # Retornar el DataFrame filtrado
    return df_resultado[mascara_habiles].copy()

In [ ]:
ph = [x for x in os.listdir(path_data+"BANREP/") if "trm" not in x]
print(len(ph))
ph

15


['agregados_crediticios.xlsx',
 'agregados_monetarios.xlsx',
 'indice_precios_vivienda.xlsx',
 'inflacion_consumidor.xlsx',
 'inflacion_consumidor2.xlsx',
 'ipp.xlsx',
 'ipp2.xlsx',
 'mercado_laboral.xlsx',
 'pib.xlsx',
 'pulso.xlsx',
 'reservas_internacionales.xlsx',
 'tasas_de_interes.xlsx',
 'tasas_de_interes2.xlsx',
 'tasas_de_interes3.xlsx',
 'unidad_valor_real.xlsx']

In [ ]:
banrep = pd.read_excel(path_data+"BANREP/trm.xlsx", decimal = ",")
banrep = banrep.iloc[1:(len(banrep)-1)].dropna(subset = "Fecha")

for i in ph:
    tmp = pd.read_excel(path_data+"BANREP/"+i, decimal = ",")
    tmp = tmp.iloc[1:(len(tmp)-1)].dropna(subset = "Fecha")
    
    banrep["Fecha"] = banrep["Fecha"].astype(str)
    tmp["Fecha"] = tmp["Fecha"].astype(str)
    banrep = pd.merge(banrep, tmp, on = "Fecha", how = "outer")

    print(f"{i}: {tmp.shape}")

print("\n", banrep.shape)
# banrep1.columns = [x.lower().replace("í","i").replace("ó","o").replace("é","e").replace(" ", "_") for x in banrep1.columns]
banrep = banrep.replace({'-':np.nan})
banrep["Fecha"] = pd.to_datetime(banrep["Fecha"], dayfirst = True)
banrep["anomes"] = banrep["Fecha"].dt.strftime("%Y-%m")

banrep = banrep.sort_values("Fecha")

for i in [x for x in banrep if x not in ["Fecha", "anomes"]]:
    try:
        banrep[i] = banrep[i].astype(float)
    except:
        banrep[i] = banrep[i].str.replace(".","").str.replace(",",".").astype(float)

banrep = banrep[banrep["Fecha"] <= max(tasascol["fecha"])]

banrep.columns = [x.replace("Tasa de Certificado de Depósito a Término","Tasa CDT").replace("Tasa de Depósitos a Término Fijo","Tasa CDT").
                   replace("Tasa Representativa del Mercado (TRM)","TRM").replace("Indicador Bancario de Referencia (IBR)","IBR").
                   replace("Índice de precios al consumidor (IPC)","IPC").replace("Tasa de política monetaria","TPM").
                   replace("Captaciones a través de CDT","Tasa CDT") for x in banrep.columns]

dels = [x for x in banrep if "Variación porcentual" in x or "Dato fin de año" in x or "Dato fin de trimestre" in x or "Dato fin de semestre" in x]
banrep.drop(columns = dels, inplace = True)
print("After dels:", banrep.shape)

banrep = calc_viernes(banrep, columna_fecha='Fecha')

banrep = banrep[["Fecha", "Fecha_viernes"] + [x for x in banrep if x not in ["Fecha", "Fecha_viernes"]]]

print("\n", banrep["Fecha"].min(), "|", banrep["Fecha"].max())
print(banrep.shape)
banrep.tail(8)

agregados_crediticios.xlsx: (1276, 6)
agregados_monetarios.xlsx: (2335, 14)
indice_precios_vivienda.xlsx: (337, 6)
inflacion_consumidor.xlsx: (855, 17)
inflacion_consumidor2.xlsx: (867, 126)
ipp.xlsx: (328, 2)
ipp2.xlsx: (681, 12)
mercado_laboral.xlsx: (309, 6)
pib.xlsx: (116, 11)
pulso.xlsx: (166, 10)
reservas_internacionales.xlsx: (801, 3)
tasas_de_interes.xlsx: (4897, 18)
tasas_de_interes2.xlsx: (9827, 23)
tasas_de_interes3.xlsx: (15689, 35)
unidad_valor_real.xlsx: (3848, 3)

 (16107, 299)
After dels: (16084, 157)

 1954-07-31 00:00:00 | 2026-10-23 00:00:00
(16084, 158)


C:\Users\PC\AppData\Local\Temp\ipykernel_16896\1419917051.py:15: Pandas4Warning: 'd' is deprecated and will be removed in a future version. Please use 'D' instead of 'd'.
  df_resultado["Fecha_viernes"] = fechas_dt + pd.to_timedelta(dias_al_viernes, unit='d')


,Fecha,Fecha_viernes,TRM(Dato diario),TRM(Dato fin de semana),TRM(Dato fin de mes),TRM(Promedio mes corrido),TRM(Promedio mensual),TRM(Promedio anual),Euro - COP/EUR - Tasa media(Dato diario),Euro - COP/EUR - Tasa media(Dato fin de semana),Euro - COP/EUR - Tasa media(Dato fin de mes),Libra esterlina - COP/GBP - Tasa media(Dato diario),Libra esterlina - COP/GBP - Tasa media(Dato fin de semana),Libra esterlina - COP/GBP - Tasa media(Dato fin de mes),"Cartera comercial en moneda legal, semanal(Dato fin de semana)","Cartera comercial en moneda extranjera , expresada en pesos, semanal(Dato fin de semana)","Cartera consumo en moneda legal, semanal(Dato fin de semana)","Cartera microcrédito en moneda legal, semanal(Dato fin de semana)","Cartera hipotecaria en moneda legal, semanal(Dato fin de semana)","Reserva Bancaria, semanal(Dato fin de semana)","Base monetaria, semanal(Dato fin de semana)","Depósitos en el sistema financiero, cuentas corrientes del sector privado, semanal(Dato fin de semana)","Depósitos en el sistema financiero, cuentas corrientes del sector público, semanal(Dato fin de semana)","Depósitos en el sistema financiero, cuentas corrientes total, semanal(Dato fin de semana)","Cuasidineros - cuentas de ahorro, semanal(Dato fin de semana)","Cuasidineros - CDT, semanal(Dato fin de semana)","Cuasidineros, total, semanal(Dato fin de semana)","Depósitos fiduciarios, semanal(Dato fin de semana)","Depósitos en el sistema financiero, cédulas, semanal(Dato fin de semana)","Depósitos en el sistema financiero, bonos, semanal(Dato fin de semana)","Depósitos en el sistema financiero, total depósitos, semanal(Dato fin de semana)","Depósitos restringidos, semanal(Dato fin de semana)",Índice de precios de la vivienda nueva (IPVNBR) - Índice real - Bogotá(Dato fin de mes),Índice de precios de la vivienda nueva (IPVNBR) - Índice real - Medellín(Dato fin de mes),Índice de precios de la vivienda nueva (IPVNBR) - Índice real - Cali(Dato fin de mes),Índice de precios de la vivienda nueva (IPVNBR) - Índice real - Agregado(Dato fin de mes),Inflación total anual(Dato fin de mes),Inflación sin alimentos(Dato fin de mes),Inflación sin alimentos ni regulados(Dato fin de mes),"Inflación de bienes, sin alimentos ni regulados(Dato fin de mes)","Inflación de servicios, sin alimentos ni regulados(Dato fin de mes)",Inflación de regulados(Dato fin de mes),"Inflación, Prendas de vestir y calzado(Dato fin de mes)","Inflación, Muebles, artículos para el hogar y para la conservación ordinaria del hogar(Dato fin de mes)","Inflación, Salud(Dato fin de mes)","Inflación, Información y comunicación(Dato fin de mes)","Inflación, Recreación y cultura(Dato fin de mes)","Inflación, Educación(Dato fin de mes)","Inflación, Restaurantes y hoteles(Dato fin de mes)","Inflación, Bienes y servicios diversos(Dato fin de mes)",Inflación de alimentos(Dato fin de mes),Inflación de alimentos y regulados(Dato fin de mes),Índice de Precios al Consumidor (IPC)(Dato fin de mes),IPC: Alimentos y bebidas no alcohólicas(Dato fin de mes),"IPC: Alojamiento, agua, electricidad, gas y otros combustibles(Dato fin de mes)","IPC: Muebles, artículos para el hogar y para la conservación ordinaria del hogar(Dato fin de mes)",IPC: Transporte(Dato fin de mes),IPC: Información y comunicación(Dato fin de mes),IPC: Educación(Dato fin de mes),IPC: Bienes y servicios diversos(Dato fin de mes),"IPC: Bogotá, D.C.(Dato fin de mes)",IPC: Cali(Dato fin de mes),IPC: Bucaramanga(Dato fin de mes),IPC: Sin alimentos(Dato fin de mes),IPC: Regulados(Dato fin de mes),IPC: de alimentos(Dato fin de mes),Índice de Precios del Productor (IPP)(Dato fin de mes)_x,Índice de Precios del Productor (IPP)(Dato fin de mes)_y,"Índice de Precios del Productor (IPP), Producción nacional - Explotación de minas y canteras(Dato fin de mes)","Índice de Precios del Productor (IPP), Oferta interna - Total(Dato fin de mes)","Índice de Precios del Productor (IPP), Oferta interna - Agricultura, ganadería, caza, silvicultura y pesca(Dato 

In [ ]:
banrep_dia = banrep[["Fecha", "Fecha_viernes"] + [x for x in banrep if "Dato diario" in x]]
print(banrep_dia.shape)

banrep_dia = calc_viernes(banrep_dia, columna_fecha='Fecha')
banrep_dia = filtrar_dias_habiles(banrep_dia, columna_fecha='Fecha')

print(banrep_dia.shape)
banrep_dia.tail(8)

C:\Users\PC\AppData\Local\Temp\ipykernel_16896\1419917051.py:15: Pandas4Warning: 'd' is deprecated and will be removed in a future version. Please use 'D' instead of 'd'.
  df_resultado["Fecha_viernes"] = fechas_dt + pd.to_timedelta(dias_al_viernes, unit='d')


(16084, 34)
(11413, 34)


,Fecha,Fecha_viernes,TRM(Dato diario),Euro - COP/EUR - Tasa media(Dato diario),Libra esterlina - COP/GBP - Tasa media(Dato diario),"IBR a 1 mes, efectiva(Dato diario)","Tasa CDT por red de oficinas, diaria(Dato diario)","Tasa CDT por tesorería, diaria(Dato diario)","Tasa CDT por red de oficinas - Monto, diaria(Dato diario)","Tasa CDT por tesorería - Monto, diaria(Dato diario)",Prime(Dato diario),"IBR overnight, efectiva(Dato diario)","IBR a 3 meses, efectiva(Dato diario)",Operaciones interbancarias - Monto(Dato diario),"IBR a 6 meses, efectiva(Dato diario)","IBR a 12 meses, efectiva(Dato diario)","Tasa CDT (CDAT) entre 15 y 29 días, diaria(Dato diario)","Tasa CDT (CDAT) a 30 días, diaria(Dato diario)","Tasa CDT (CDAT) entre 31 y 90 días, diaria(Dato diario)","Tasa CDT (CDAT) entre 91 y 180 días, diaria(Dato diario)","Tasa CDT (CDT) a 120 días - Monto, diaria(Dato diario)_x","Tasa CDT (CDAT) entre 15 y 29 días - Monto, diaria(Dato diario)","Tasa CDT (CDAT) a 30 días - Monto, diaria(Dato diario)","Tasa CDT (CDAT) entre 31 y 90 días - Monto, diaria(Dato diario)","Tasa CDT (CDAT) entre 91 y 180 días - Monto, diaria(Dato diario)",TPM(Dato diario),"Tasa CDT (DTF) a 90 días, semanal(Dato diario)",Tasa interbancaria (TIB)(Dato diario),"Tasa CDT (CDT) a 360 días, diaria(Dato diario)","Tasa CDT (CDT) a 30 días, diaria(Dato diario)","Tasa CDT (CDT) a 120 días, diaria(Dato diario)","Tasa CDT (CDT) a 30 días - Monto, diaria(Dato diario)","Tasa CDT (CDT) a 120 días - Monto, diaria(Dato diario)_y","Tasa CDT (CDT) a 360 días - Monto, diaria(Dato diario)"
7164,2026-10-14,2026-10-16,3181.7400,3538.5721,4186.6926,12.2960,10.4400,12.8400,1627857.4400,638473.1900,7.0000,12.2640,12.4100,1110349.2900,12.5090,12.6790,1.4000,0.5800,1.9900,10.0400,72297.1100,41500.0000,930.0000,2904.0000,86785.0000,12.2500,10.3900,12.2400,12.6000,9.2300,10.6800,234548.4700,72297.1100,505870.9700
7681,2026-10-15,2026-10-16,3197.6500,3581.6878,4237.3659,12.3320,10.7000,12.9200,1613529.3500,398528.0600,7.0000,12.2600,12.3880,1207853.0800,12.4810,12.5990,NaN,0.6000,9.0400,10.1000,107435.0200,NaN,1284.0000,9945.0000,99649.0000,12.2500,10.3900,12.2500,12.4100,9.6500,11.1500,182729.3100,107435.0200,293766.2900
8200,2026-10-16,2026-10-16,3170.4600,3567.7186,4233.5152,12.2950,10.6700,12.8400,1893511.5600,432575.6900,7.0000,12.2570,12.3820,1391143.1900,12.4680,12.5630,NaN,0.3000,7.3600,10.0400,195130.3500,NaN,24259.0000,6294.0000,77952.0000,NaN,10.3900,12.2400,12.3100,9.6900,11.7200,189675.4600,195130.3500,193017.4500
9752,2026-10-19,2026-10-23,3174.6700,3561.0273,4227.8668,12.3080,10.4100,12.9600,2431194.5200,431695.8400,7.0000,12.2710,12.4210,1211591.8000,12.5110,12.6210,0.9500,NaN,2.0200,10.0600,241587.3200,20.0000,NaN,14217.0000,96378.0000,12.2500,10.4400,12.2500,12.3000,9.5000,11.2100,456467.6500,241587.3200,304277.0000
10270,2026-10-20,2026-10-23,3156.3200,3549.4396,4225.5234,12.3020,10.5900,13.1100,1251252.4800,639377.9200,7.0000,12.2450,12.4120,1378293.0300,12.4840,12.5870,1.4000,0.5000,0.8200,10.1900,58035.4300,6560.0000,496.0000,4627.0000,63207.0000,12.2500,10.4400,12.2000,12.6500,9.6300,10.4300,165705.5500,58035.4300,296417.5900
10787,2026-10-21,2026-10-23,3163.7600,3551.0042,4234.3764,12.3060,10.6400,12.9300,1781563.6200,981711.1500,7.0000,12.2590,12.4360,1279246.0100,12.5250,12.5980,NaN,0.3700,1.9000,10.1200,51873.8700,NaN,60.0000,3674.0000,73149.0000,12.2500,10.4400,12.2400,12.6800,9.6400,10.4200,186353.3100,51873.8700,578722.4800
11304,2026-10-22,2026-10-23,3191.4700,3573.6485,4265.2401,12.3090,10.8800,12.8600,1394934.5200,709293.6400,7.0000,12.2580,12.4380,1235634.6700,12.5170,12.6130,NaN,NaN,9.1800,10.1000,63704.1900,NaN,NaN,4780.0000,63663.0000,12.2500,10.4400,12.2500,12.7800,9.6900,10.4900,154674.9000,63704.1900,324297.1100
11822,2026-10-23,2026-10-23,3185.0100,3568.4852,4263.2951,12.3680,10.8700,12.9600,1508445.3600,573801.3600,7.0000,12.2670,12.4400,1396098.9800,12.5380,12.6070,0.7900,0.4000,4.6200,10.0100,61620.8600,4240.0000,240.0000,1695.0000,1005.9100,12

In [ ]:
banrep_seman = banrep[["Fecha", "Fecha_viernes"] + [x for x in banrep if "Dato semanal" in x]]
cols = [x for x in banrep_seman if x not in ["Fecha", "Fecha_viernes"]] 
banrep_seman = banrep_seman.dropna(subset=cols, how='all')

banrep_seman = calc_viernes(banrep_seman, columna_fecha='Fecha')
banrep_seman = filtrar_dias_habiles(banrep_seman, columna_fecha='Fecha')

print(banrep_seman.shape)
banrep_seman.tail(8)

(2232, 19)

C:\Users\PC\AppData\Local\Temp\ipykernel_16896\1419917051.py:15: Pandas4Warning: 'd' is deprecated and will be removed in a future version. Please use 'D' instead of 'd'.
  df_resultado["Fecha_viernes"] = fechas_dt + pd.to_timedelta(dias_al_viernes, unit='d')


,Fecha,Fecha_viernes,"Construcción de vivienda diferente de VIS (colocación en COP), Tasa de interés, semanal(Dato semanal)","Construcción de vivienda VIS (colocación en COP), Tasa de interés, semanal(Dato semanal)","Adquisición de vivienda diferente de VIS (colocación en COP), Tasa de interés, semanal(Dato semanal)","Adquisición de vivienda VIS (colocación en COP), Tasa de interés, semanal(Dato semanal)","Crédito de consumo, Tasa de interés, semanal(Dato semanal)","Créditos Comerciales (Ordinario), Tasa de interés, semanal(Dato semanal)","Créditos Comerciales (Preferencial o Corporativo), Tasa de interés, semanal(Dato semanal)","Créditos Comerciales (Tesorería), Tasa de interés, semanal(Dato semanal)","Crédito popular productivo (total), semanal(Dato semanal)","Crédito productivo (total), semanal(Dato semanal)","Crédito productivo de mayor monto(total), semanal(Dato semanal)","Microcréditos (diferentes de leasing), Tasa de interés, semanal(Dato semanal)",TPM(Dato semanal),"Tasa CDT (DTF) a 90 días, semanal(Dato semanal)","Tasa de interés de colocación Banco de la República, semanal(Dato semanal)","Tasa de interés de colocación Total, semanal(Dato semanal)","Tasa de interés de colocación sin Tesorería, semanal(Dato semanal)"
1937,2026-09-04,2026-09-04,15.2700,15.1100,15.3700,14.2900,21.3400,15.4600,13.7500,13.5100,54.0600,30.4200,26.5700,NaN,12.0000,10.1900,16.6000,16.5500,16.6100
5567,2026-09-11,2026-09-11,15.3000,15.4300,15.3300,14.3800,21.3000,14.8400,14.0100,13.5200,55.2100,31.0700,27.2200,NaN,12.0000,10.0500,16.3300,16.2800,16.3400
9191,2026-09-18,2026-09-18,15.2600,15.0700,15.3800,14.3400,21.4000,15.5800,14.6800,13.1500,55.7500,33.6800,28.1500,NaN,12.0000,10.2200,16.7700,16.7100,16.7800
12813,2026-09-25,2026-09-25,15.3700,15.1800,15.3800,14.4700,20.9600,15.2500,14.6300,13.0700,54.2200,32.8600,28.0200,NaN,12.0000,10.2600,16.4600,16.4000,16.4800
947,2026-10-02,2026-10-02,15.1800,15.1600,15.2700,14.1000,20.9500,15.1500,13.6800,12.7500,53.7400,32.8200,27.8200,NaN,12.2500,10.3800,16.2500,16.1400,16.2700
4576,2026-10-09,2026-10-09,15.5100,15.3600,15.3500,14.3800,21.5900,15.8300,15.0100,13.9400,56.3700,34.0900,28.2400,NaN,12.2500,10.1900,17.2400,17.1800,17.2500
8200,2026-10-16,2026-10-16,15.3300,15.2500,15.3300,14.4400,21.6600,15.8400,15.0100,13.4900,55.9900,34.7100,28.4800,NaN,NaN,10.3900,17.3200,17.2100,17.3400
11822,2026-10-23,2026-10-23,15.4100,15.2200,15.3600,14.3300,21.2500,14.9600,14.7500,13.8300,56.6500,35.2100,28.5100,NaN,12.2500,10.4400,16.4300,16.3700,16.4500


In [ ]:
banrep_mes = banrep[["Fecha", "Fecha_viernes"] + [x for x in banrep if "Dato fin de mes" in x]]
cols = [x for x in banrep_mes if x not in ["Fecha", "Fecha_viernes"]] 
banrep_mes = banrep_mes.dropna(subset=cols, how='all')

print(banrep_mes.shape)
banrep_mes.tail(8)

(868, 71)


,Fecha,Fecha_viernes,TRM(Dato fin de mes),Euro - COP/EUR - Tasa media(Dato fin de mes),Libra esterlina - COP/GBP - Tasa media(Dato fin de mes),Índice de precios de la vivienda nueva (IPVNBR) - Índice real - Bogotá(Dato fin de mes),Índice de precios de la vivienda nueva (IPVNBR) - Índice real - Medellín(Dato fin de mes),Índice de precios de la vivienda nueva (IPVNBR) - Índice real - Cali(Dato fin de mes),Índice de precios de la vivienda nueva (IPVNBR) - Índice real - Agregado(Dato fin de mes),Inflación total anual(Dato fin de mes),Inflación sin alimentos(Dato fin de mes),Inflación sin alimentos ni regulados(Dato fin de mes),"Inflación de bienes, sin alimentos ni regulados(Dato fin de mes)","Inflación de servicios, sin alimentos ni regulados(Dato fin de mes)",Inflación de regulados(Dato fin de mes),"Inflación, Prendas de vestir y calzado(Dato fin de mes)","Inflación, Muebles, artículos para el hogar y para la conservación ordinaria del hogar(Dato fin de mes)","Inflación, Salud(Dato fin de mes)","Inflación, Información y comunicación(Dato fin de mes)","Inflación, Recreación y cultura(Dato fin de mes)","Inflación, Educación(Dato fin de mes)","Inflación, Restaurantes y hoteles(Dato fin de mes)","Inflación, Bienes y servicios diversos(Dato fin de mes)",Inflación de alimentos(Dato fin de mes),Inflación de alimentos y regulados(Dato fin de mes),Índice de Precios al Consumidor (IPC)(Dato fin de mes),IPC: Alimentos y bebidas no alcohólicas(Dato fin de mes),"IPC: Alojamiento, agua, electricidad, gas y otros combustibles(Dato fin de mes)","IPC: Muebles, artículos para el hogar y para la conservación ordinaria del hogar(Dato fin de mes)",IPC: Transporte(Dato fin de mes),IPC: Información y comunicación(Dato fin de mes),IPC: Educación(Dato fin de mes),IPC: Bienes y servicios diversos(Dato fin de mes),"IPC: Bogotá, D.C.(Dato fin de mes)",IPC: Cali(Dato fin de mes),IPC: Bucaramanga(Dato fin de mes),IPC: Sin alimentos(Dato fin de mes),IPC: Regulados(Dato fin de mes),IPC: de alimentos(Dato fin de mes),Índice de Precios del Productor (IPP)(Dato fin de mes)_x,Índice de Precios del Productor (IPP)(Dato fin de mes)_y,"Índice de Precios del Productor (IPP), Producción nacional - Explotación de minas y canteras(Dato fin de mes)","Índice de Precios del Productor (IPP), Oferta interna - Total(Dato fin de mes)","Índice de Precios del Productor (IPP), Oferta interna - Agricultura, ganadería, caza, silvicultura y pesca(Dato fin de mes)","Índice de Precios del Productor (IPP), Oferta interna - Industria manufactureras(Dato fin de mes)","Índice de Precios del Productor (IPP), Según procedencia - producidos y consumidos(Dato fin de mes)","Índice de Precios del Productor (IPP), Según procedencia - Importados(Dato fin de mes)","Índice de Precios del Productor (IPP), Según procedencia - Exportados(Dato fin de mes)","Índice de Precios del Productor (IPP), Según uso o destino económico - Consumo intermedio(Dato fin de mes)","Índice de Precios del Productor (IPP), Según uso o destino económico - Consumo final(Dato fin de mes)","Índice de Precios del Productor (IPP), Según uso o destino económico - Bienes de capital(Dato fin de mes)",Tasa global de participación(Dato fin de mes),"Tasa de ocupación, según nivel educativo: sin educación superior(Dato fin de mes)","Tasa de desempleo, según nivel educativo: sin educación superior(Dato fin de mes)",Tasa de desempleo - total nacional(Dato fin de mes),Tasa de ocupación - total nacional(Dato fin de mes),Pulso Económico Regional - Comercio(Dato fin de mes),Pulso Económico Regional - Transporte(Dato fin de mes),Pulso Económico Regional - Agropecuario(Dato fin de mes),Pulso Económico Regional - Financiero(Dato fin de mes),Pulso Económico Regional - Vivienda(Dato fin de mes),Pulso Económico Regional - Nacional(Dato fin de mes),Pulso Económico Regional - Antioquia(Dato fin de mes),Pulso Económico Regional - Suroccidente(Dato fin de mes),Pulso Económico Regional - Bogotá(Dato fin de mes),Reservas internacionales brutas(Dato fin d

In [ ]:
falta trimestral -> solo pib

SyntaxError: invalid syntax (180304931.py, line 1)

In [ ]:
falta anual -> pib, salario minimo

## FED rate

In [ ]:
fedrate = pd.read_csv(path_data + "FED_rate.csv", sep = ",", decimal = ",")
fedrate = fedrate.rename(columns = {"observation_date":"fecha", "FEDFUNDS":"FED rate"})
fedrate["fecha"] = pd.to_datetime(fedrate["fecha"])
print(fedrate.shape)
print(min(fedrate["fecha"]))
print(max(fedrate["fecha"]))
fedrate

(322, 2)
2000-01-01 00:00:00
2026-10-01 00:00:00


,fecha,FED rate
0,2000-01-01,5.45
1,2000-02-01,5.73
2,2000-03-01,5.85
3,2000-04-01,6.02
4,2000-05-01,6.27
...,...,...
317,2026-06-01,3.63
318,2026-07-01,3.63
319,2026-08-01,3.63
320,2026-09-01,3.75


## FED

In [ ]:
fed = pd.read_csv(path_data + "FED.csv", sep = ",", decimal = ",")
fed = fed.iloc[5:].rename(columns = {"Series Description":"fecha"}).replace({"ND":np.nan})
fed["fecha"] = pd.to_datetime(fed["fecha"])
cols = [x for x in fed if x not in ["fecha"]]
fed[cols] = fed[cols].astype(float)
print(fed.shape)
print(min(fed["fecha"]))
print(max(fed["fecha"]))
fed.tail(5)

(1401, 28)
2000-01-07 00:00:00
2026-11-06 00:00:00


,fecha,30-Day AA Nonfinancial Commercial Paper Interest Rate,60-Day AA Nonfinancial Commercial Paper Interest Rate,90-Day AA Nonfinancial Commercial Paper Interest Rate,30-Day AA Financial Commercial Paper Interest Rate,60-Day AA Financial Commercial Paper Interest Rate,90-Day AA Financial Commercial Paper Interest Rate,4-week Treasury bill secondary market rate discount basis,3-month Treasury bill secondary market rate discount basis,6-month Treasury bill secondary market rate discount basis,1-year Treasury bill secondary market rate^ discount basis,"Market yield on U.S. Treasury securities at 1-month constant maturity, quoted on investment basis","Market yield on U.S. Treasury securities at 3-month constant maturity, quoted on investment basis","Market yield on U.S. Treasury securities at 6-month constant maturity, quoted on investment basis","Market yield on U.S. Treasury securities at 1-year constant maturity, quoted on investment basis","Market yield on U.S. Treasury securities at 2-year constant maturity, quoted on investment basis","Market yield on U.S. Treasury securities at 3-year constant maturity, quoted on investment basis","Market yield on U.S. Treasury securities at 5-year constant maturity, quoted on investment basis","Market yield on U.S. Treasury securities at 7-year constant maturity, quoted on investment basis","Market yield on U.S. Treasury securities at 10-year constant maturity, quoted on investment basis","Market yield on U.S. Treasury securities at 20-year constant maturity, quoted on investment basis","Market yield on U.S. Treasury securities at 30-year constant maturity, quoted on investment basis","Market yield on U.S. Treasury securities at 5-year constant maturity, quoted on investment basis, inflation-indexed","Market yield on U.S. Treasury securities at 7-year constant maturity, quoted on investment basis, inflation-indexed","Market yield on U.S. Treasury securities at 10-year constant maturity, quoted on investment basis, inflation-indexed","Market yield on U.S. Treasury securities at 20-year constant maturity, quoted on investment basis, inflation-indexed","Market yield on U.S. Treasury securities at 30-year^ constant maturity, quoted on investment basis,^ inflation-indexed",Treasury long-term average (over 10 years)
1401,2026-10-09,3.9500,4.0000,4.0700,3.9400,4.0100,4.0800,3.9100,4.0500,4.1600,4.2400,4.0900,4.2300,4.3000,4.4500,4.7900,4.8900,5.0300,5.1500,5.2600,5.6800,5.6300,2.6600,2.7800,2.9100,3.1900,3.3400,3.3400
1402,2026-10-16,3.9700,4.0200,4.0900,NaN,4.0700,4.1100,3.9500,4.0900,4.2100,4.2700,4.1300,4.2700,4.3500,4.4900,4.7800,4.8700,4.9500,5.0500,5.1500,5.5300,5.4900,2.6000,2.7000,2.8100,3.0700,3.2200,3.2100
1403,2026-10-23,3.9600,4.0500,NaN,3.9400,4.0800,4.0800,3.9300,4.0700,4.2000,4.2500,4.1000,4.2400,4.3300,4.4800,4.7400,4.8100,4.8900,4.9700,5.0700,5.4300,5.3900,2.4700,2.5900,2.7300,2.9900,3.1300,3.1200
1404,2026-10-30,3.9800,4.0700,NaN,NaN,4.0100,4.1600,3.9200,4.0200,4.1700,4.2400,4.0800,4.1900,4.3100,4.4400,4.6400,4.7500,4.8000,4.8900,4.9900,5.3600,5.3200,2.3300,2.4800,2.6400,2.9300,3.0700,3.0600
1405,2026-11-06,3.9800,4.1100,4.1100,NaN,4.0300,4.0700,3.8800,4.0200,4.1300,4.1600,4.0300,4.2000,4.2400,4.3600,4.5500,4.6200,4.6700,4.7700,4.8900,5.3000,5.2800,2.3500,2.4800,2.6300,2.9200,3.0700,3.0600


## European Central Bank

In [ ]:
ecb = pd.read_csv(path_data + "ECB.csv", sep = ",")
ecb = ecb.rename(columns = {"DATE":"fecha", "Main refinancing operations - fixed rate tenders (fixed rate) (date of changes) - Level (FM.B.U2.EUR.4F.KR.MRR_FR.LEV)":"ECB rate"})
ecb["fecha"] = pd.to_datetime(ecb["fecha"])
del ecb["TIME PERIOD"]

print(ecb.shape)
print(min(ecb["fecha"]))
print(max(ecb["fecha"]))

ecb.tail(10)

(48, 2)
1999-01-01 00:00:00
2026-09-16 00:00:00


,fecha,ECB rate
38,2024-06-12,4.2500
39,2024-09-18,3.6500
40,2024-10-23,3.4000
41,2024-12-18,3.1500
42,2025-02-05,2.9000
43,2025-03-12,2.6500
44,2025-04-23,2.4000
45,2025-06-11,2.1500
46,2026-06-17,2.4000
47,2026-09-16,2.6500


## MSCI Colombia

In [ ]:
esta desde 2012

SyntaxError: invalid syntax (1791038506.py, line 1)

In [ ]:
msci = pd.read_csv(path_data + "MSCI_Colombia.csv", sep = ",", decimal = ".", thousands=",")
msci = msci.rename(columns = {"Fecha":"fecha", "Último":"MSCI_close", "Apertura":"MSCI_open", "% var.":"vari_MSCI"})
msci["fecha"] = pd.to_datetime(msci["fecha"], dayfirst = True)
msci = msci[["fecha", "MSCI_close", "MSCI_open", "vari_MSCI"]].sort_values("fecha")

for i in ["MSCI_close", "MSCI_open", "vari_MSCI"]:
    msci[i] = msci[i].str.replace(".","").str.replace(",",".").str.replace("%","").astype(float)

print(msci.shape)
print(min(msci["fecha"]))
print(max(msci["fecha"]))

msci.tail(10)

(3729, 4)
2012-07-09 00:00:00
2026-11-09 00:00:00


,fecha,MSCI_close,MSCI_open,vari_MSCI
3719,2026-10-27,1182.8000,1200.6900,0.2500
3720,2026-10-28,1176.1400,1183.6800,-0.5600
3721,2026-10-29,1164.7100,1179.1100,-0.9700
3722,2026-10-30,1150.3800,1162.1200,-1.2300
3723,2026-11-02,1152.2300,1152.2300,0.1600
3724,2026-11-03,1160.5700,1151.3000,0.7200
3725,2026-11-04,1172.3200,1167.7300,1.0100
3726,2026-11-05,1151.2900,1174.2900,-1.7900
3727,2026-11-06,1139.1500,1147.4400,-1.0500
3728,2026-11-09,1178.4200,1137.1500,3.4500


## Yahoo

In [ ]:
import yfinance as yf

dicc = {
# Divisas
"EURUSD=X":"euro_dolar", 
"GBPUSD=X":"libra_dolar", 
"JPY=X":"yen_dolar",
"CNY=X":"yuan_dolar", 
"COP=X": "usd_cop",
"DX-Y.NYB": "dxy_dolar",
# Tasas de Bonos del Tesoro de EE.UU.
"^IRX":"bonos_usa_1y", 
"^FVX":"bonos_usa_5y", 
"^TNX":"bonos_usa_10y", 
"^TYX":"bonos_usa_30y",
# Índices bursátiles globales
"^GSPC": "sp500",
"^IXIC": "nasdaq",
"^DJI": "dow_jones",
"EEM": "msci_emergentes",
"URTH": "msci_world",
# Índices de volatilidad / riesgo
"^VIX": "vix",
# Petróleo
"CL=F": "petrol_wti",
"BZ=F": "petrol_brent",
# Metales
"GC=F": "oro",
"SI=F": "plata",
"HG=F": "cobre",
# Energía
"NG=F": "gas_natural",
# Tasas internacionales / futuros
"SR1=F": "sofr_futuro_1m",
"GE=F": "eurodolar_futuro",
# ETFs de bonos internacionales
"TLT": "bonos_largo_usa",
"SHY": "bonos_corto_usa",
"LQD": "bonos_corps_inversion",
"HYG": "bonos_high_yield",
"EMB": "bonos_mercados_emergentes",
# Criptomonedas (liquidez global)
"BTC-USD": "bitcoin",
"ETH-USD": "ethereum"
}
yahoo = pd.DataFrame(bpmensual["anomes"])

for sym, name in dicc.items():
    #print(name)
    _temp = yf.Ticker(sym)
    _temp = _temp.history(period="10y")
    _temp = _temp.reset_index()
    _temp.rename(columns = {"Date":"fecha", "Close":name}, inplace = True)
    _temp["fecha"] = pd.to_datetime(_temp["fecha"])
    _temp["fecha"] = _temp["fecha"].dt.strftime("%Y-%m")
    _temp = _temp.sort_values("fecha", ascending = False).drop_duplicates("fecha")[["fecha", name]]
    #_temp = _temp.groupby("fecha")[name].mean().reset_index()
    yahoo = pd.merge(yahoo, _temp, on="fecha", how="left")

print(min(yahoo["fecha"]))
print(max(yahoo["fecha"]))

yahoo

## Captaciones

In [ ]:
tasascap = pd.read_csv(path_data + "tasas_de_interes_captacion.csv", decimal = ",")
tasascap.columns = [x.lower() for x in tasascap.columns]
tasascap.rename(columns = {"fechacorte":"fecha"}, inplace = True)
tasascap["fecha"] = pd.to_datetime(tasascap["fecha"], dayfirst = True)

print(tasascap.shape)
print(min(tasascap["fecha"]))
print(max(tasascap["fecha"]))

tasascap.tail(5)

(2105369, 10)
2018-01-31 00:00:00
2026-11-06 00:00:00


,tipoentidad,codigoentidad,nombreentidad,fecha,uca,nombre_unidad_de_captura,subcuenta,descripcion,tasa,monto
2105364,32,1,Cooperativa Financiera de Antioquia,2026-11-06,8,SALDO DE CTAS CTES CDT Y CDAT,90,SALDO EN CUENTAS DE AHORRO AFC EN PESOS,0.0000,"299.680,04"
2105365,32,2,Cooperativa Financiera JFK,2026-11-06,1,EMISIONES PUNTUALES Y RANGOS DE EMISION DE CDT,10,A 30 DIAS,4.5000,"272.221,46"
2105366,1,66,Banco BTG Pactual Colombia S.A.,2026-11-06,1,EMISIONES PUNTUALES Y RANGOS DE EMISION DE CDT,910,CAPTACIONES A TRAVES DE CDT POR TESORERIA,13.0900,3.000.000
2105367,32,2,Cooperativa Financiera JFK,2026-11-06,1,EMISIONES PUNTUALES Y RANGOS DE EMISION DE CDT,60,ENTRE 61 Y 89 DIAS,6.4800,41.600
2105368,32,2,Cooperativa Financiera JFK,2026-11-06,8,SALDO DE CTAS CTES CDT Y CDAT,50,SALDO EN DEPOSITOS DE AHORRO INACTIVOS Y ABANDONADOS PERSONA JURIDICA,0.0000,"114.173.078,84"


## Cartera vencida

In [ ]:
venci = pd.read_csv(path_data + "cartera_vencida.csv", decimal = ",")
venci.columns = [x.lower() for x in venci.columns]
venci.rename(columns = {"fecha_corte":"fecha"}, inplace = True)
venci["fecha"] = pd.to_datetime(venci["fecha"], dayfirst = True)

print(venci.shape)
print(min(venci["fecha"]))
print(max(venci["fecha"]))

venci.tail(4)

C:\Users\PC\AppData\Local\Temp\ipykernel_7192\4033505613.py:1: DtypeWarning: Columns (0: (3) Vencida 1-2 Meses, 1: (4) Vencida 2-3 Meses, 2: (5) Vencida 1-3 Meses, 3: (6) Vencida 3-4 Meses, 4: (7) Vencida > de 4 Meses, 5: (9) Vencida +6 meses, 6: (10) Vencida 1-4 meses, 7: (11) Vencida 4-6 meses, 8: (12) Vencida 6-12 meses, 9: (13) Vencida 12-18 meses, 10: (14) Vencida > 12 meses, 11: (15) Vencida > 18 meses) have mixed types. Specify dtype option on import or set low_memory=False.
  venci = pd.read_csv(path_data + "cartera_vencida.csv", decimal = ",")


(112602, 34)
2015-01-31 00:00:00
2026-08-31 00:00:00


,tipo_entidad,codigo_entidad,nombreentidad,fecha,unicap,descrip_uc,renglon,desc_renglon,(1) saldo de la cartera a la fecha de corte del reporte,(2) vigente,(3) vencida 1-2 meses,(4) vencida 2-3 meses,(5) vencida 1-3 meses,(6) vencida 3-4 meses,(7) vencida > de 4 meses,(8) vencida 3-6 meses,(9) vencida +6 meses,(10) vencida 1-4 meses,(11) vencida 4-6 meses,(12) vencida 6-12 meses,(13) vencida 12-18 meses,(14) vencida > 12 meses,(15) vencida > 18 meses,(16) número de clientes mora > 30 días,(17) calificación de riesgo a / número de clientes,(18) calificación de riesgo a / saldo,(19) calificación de riesgo b / número de clientes,(20) calificación de riesgo b / saldo,(21) calificación de riesgo c / número de clientes,(22) calificación de riesgo c / saldo,(23) calificación de riesgo d / número de clientes,(24) calificación de riesgo d / saldo,(25) calificación de riesgo e / número de clientes,(26) calificación de riesgo e / saldo
112598,32,4,Cotrafa Financiera,2026-08-31,32,CREDITOS DE VIVIENDA PARA EMPL NO VIS,5,CRÉDITO EMPLEADOS VIVIENDA NO VIS,9.208.278.866,9.208.278.866,0,0,0,0,0,0,0,0,0,0,0,0,0,0,105,9.208.278.866,0,0,0,0,0,0,0,0
112599,32,2,Jfk Cooperativa Financiera,2026-08-31,32,CREDITOS DE VIVIENDA PARA EMPL NO VIS,5,CRÉDITO EMPLEADOS VIVIENDA NO VIS,514.331.283,514.331.283,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,514.331.283,0,0,0,0,0,0,0,0
112600,32,5,Confiar Cooperativa Financiera,2026-08-31,32,CREDITOS DE VIVIENDA PARA EMPL NO VIS,5,CRÉDITO EMPLEADOS VIVIENDA NO VIS,5.868.437.692,5.868.437.692,0,0,0,0,0,0,0,0,0,0,0,0,0,0,40,5.868.437.692,0,0,0,0,0,0,0,0
112601,32,1,Cooperativa Financiera De Antioquia Cfa,2026-08-31,32,CREDITOS DE VIVIENDA PARA EMPL NO VIS,5,CRÉDITO EMPLEADOS VIVIENDA NO VIS,4.537.635.049,4.537.635.049,0,0,0,0,0,0,0,0,0,0,0,0,0,0,34,4.537.635.049,0,0,0,0,0,0,0,0


In [ ]:
venci["FECHA_CORTE"].describe()

count                        112602
mean     2020-11-12 01:08:23.543454
min             2015-01-31 00:00:00
25%             2017-11-30 00:00:00
50%             2020-10-31 00:00:00
75%             2023-10-31 00:00:00
max             2026-08-31 00:00:00
Name: FECHA_CORTE, dtype: object

## Merge

In [ ]:
# Mensual
tasascol["anomes"] = tasascol["anomes"].astype(str)
banrep_mes["anomes"] = banrep_mes["Fecha"].astype(str).str[:7]

print(tasascol.shape)
df = pd.merge(tasascol, banrep_mes.drop(columns = ["Fecha", "Fecha_viernes"]), left_on = "anomes", right_on = "anomes", how = "left", indicator = True)
display(utils.vc(df["_merge"]))
print(df.shape)


# Semanal



# Diario



# Trimestal


# Anual



# FED rate



# FED - Other variables



# Cartera vencida



# Yahoo





df = df.sort_values("fecha")

df.tail(5)

(6942, 9)


,_merge,count,porc
0,both,6939,99.5981
1,left_only,28,0.4019
2,right_only,0,0.0000


(6967, 79)


,fecha,producto,tasa,monto,anomes,vari_monto_m1,vari_tasa_m1,y_vari_anual_monto,vari_anual_tasa,TRM(Dato fin de mes),Euro - COP/EUR - Tasa media(Dato fin de mes),Libra esterlina - COP/GBP - Tasa media(Dato fin de mes),Índice de precios de la vivienda nueva (IPVNBR) - Índice real - Bogotá(Dato fin de mes),Índice de precios de la vivienda nueva (IPVNBR) - Índice real - Medellín(Dato fin de mes),Índice de precios de la vivienda nueva (IPVNBR) - Índice real - Cali(Dato fin de mes),Índice de precios de la vivienda nueva (IPVNBR) - Índice real - Agregado(Dato fin de mes),Inflación total anual(Dato fin de mes),Inflación sin alimentos(Dato fin de mes),Inflación sin alimentos ni regulados(Dato fin de mes),"Inflación de bienes, sin alimentos ni regulados(Dato fin de mes)","Inflación de servicios, sin alimentos ni regulados(Dato fin de mes)",Inflación de regulados(Dato fin de mes),"Inflación, Prendas de vestir y calzado(Dato fin de mes)","Inflación, Muebles, artículos para el hogar y para la conservación ordinaria del hogar(Dato fin de mes)","Inflación, Salud(Dato fin de mes)","Inflación, Información y comunicación(Dato fin de mes)","Inflación, Recreación y cultura(Dato fin de mes)","Inflación, Educación(Dato fin de mes)","Inflación, Restaurantes y hoteles(Dato fin de mes)","Inflación, Bienes y servicios diversos(Dato fin de mes)",Inflación de alimentos(Dato fin de mes),Inflación de alimentos y regulados(Dato fin de mes),Índice de Precios al Consumidor (IPC)(Dato fin de mes),IPC: Alimentos y bebidas no alcohólicas(Dato fin de mes),"IPC: Alojamiento, agua, electricidad, gas y otros combustibles(Dato fin de mes)","IPC: Muebles, artículos para el hogar y para la conservación ordinaria del hogar(Dato fin de mes)",IPC: Transporte(Dato fin de mes),IPC: Información y comunicación(Dato fin de mes),IPC: Educación(Dato fin de mes),IPC: Bienes y servicios diversos(Dato fin de mes),"IPC: Bogotá, D.C.(Dato fin de mes)",IPC: Cali(Dato fin de mes),IPC: Bucaramanga(Dato fin de mes),IPC: Sin alimentos(Dato fin de mes),IPC: Regulados(Dato fin de mes),IPC: de alimentos(Dato fin de mes),Índice de Precios del Productor (IPP)(Dato fin de mes)_x,Índice de Precios del Productor (IPP)(Dato fin de mes)_y,"Índice de Precios del Productor (IPP), Producción nacional - Explotación de minas y canteras(Dato fin de mes)","Índice de Precios del Productor (IPP), Oferta interna - Total(Dato fin de mes)","Índice de Precios del Productor (IPP), Oferta interna - Agricultura, ganadería, caza, silvicultura y pesca(Dato fin de mes)","Índice de Precios del Productor (IPP), Oferta interna - Industria manufactureras(Dato fin de mes)","Índice de Precios del Productor (IPP), Según procedencia - producidos y consumidos(Dato fin de mes)","Índice de Precios del Productor (IPP), Según procedencia - Importados(Dato fin de mes)","Índice de Precios del Productor (IPP), Según procedencia - Exportados(Dato fin de mes)","Índice de Precios del Productor (IPP), Según uso o destino económico - Consumo intermedio(Dato fin de mes)","Índice de Precios del Productor (IPP), Según uso o destino económico - Consumo final(Dato fin de mes)","Índice de Precios del Productor (IPP), Según uso o destino económico - Bienes de capital(Dato fin de mes)",Tasa global de participación(Dato fin de mes),"Tasa de ocupación, según nivel educativo: sin educación superior(Dato fin de mes)","Tasa de desempleo, según nivel educativo: sin educación superior(Dato fin de mes)",Tasa de desempleo - total nacional(Dato fin de mes),Tasa de ocupación - total nacional(Dato fin de mes),Pulso Económico Regional - Comercio(Dato fin de mes),Pulso Económico Regional - Transporte(Dato fin de mes),Pulso Económico Regional - Agropecuario(Dato fin de mes),Pulso Económico Regional - Financiero(Dato fin de mes),Pulso Económico Regional - Vivienda(Dato fin de mes),Pulso Económico Regional - Nacional(Dato fin de mes),Pulso Económico Regional - Antioquia(Dato fin de mes),Pulso Económico Regional - Suroccidente(Dato fin de mes),Pulso Económico Re

In [ ]:
df.iloc[:len(df)-28].tail(5)

,fecha,producto,tasa,monto,anomes,vari_monto_m1,vari_tasa_m1,y_vari_anual_monto,vari_anual_tasa,TRM(Dato fin de mes),Euro - COP/EUR - Tasa media(Dato fin de mes),Libra esterlina - COP/GBP - Tasa media(Dato fin de mes),Índice de precios de la vivienda nueva (IPVNBR) - Índice real - Bogotá(Dato fin de mes),Índice de precios de la vivienda nueva (IPVNBR) - Índice real - Medellín(Dato fin de mes),Índice de precios de la vivienda nueva (IPVNBR) - Índice real - Cali(Dato fin de mes),Índice de precios de la vivienda nueva (IPVNBR) - Índice real - Agregado(Dato fin de mes),Inflación total anual(Dato fin de mes),Inflación sin alimentos(Dato fin de mes),Inflación sin alimentos ni regulados(Dato fin de mes),"Inflación de bienes, sin alimentos ni regulados(Dato fin de mes)","Inflación de servicios, sin alimentos ni regulados(Dato fin de mes)",Inflación de regulados(Dato fin de mes),"Inflación, Prendas de vestir y calzado(Dato fin de mes)","Inflación, Muebles, artículos para el hogar y para la conservación ordinaria del hogar(Dato fin de mes)","Inflación, Salud(Dato fin de mes)","Inflación, Información y comunicación(Dato fin de mes)","Inflación, Recreación y cultura(Dato fin de mes)","Inflación, Educación(Dato fin de mes)","Inflación, Restaurantes y hoteles(Dato fin de mes)","Inflación, Bienes y servicios diversos(Dato fin de mes)",Inflación de alimentos(Dato fin de mes),Inflación de alimentos y regulados(Dato fin de mes),Índice de Precios al Consumidor (IPC)(Dato fin de mes),IPC: Alimentos y bebidas no alcohólicas(Dato fin de mes),"IPC: Alojamiento, agua, electricidad, gas y otros combustibles(Dato fin de mes)","IPC: Muebles, artículos para el hogar y para la conservación ordinaria del hogar(Dato fin de mes)",IPC: Transporte(Dato fin de mes),IPC: Información y comunicación(Dato fin de mes),IPC: Educación(Dato fin de mes),IPC: Bienes y servicios diversos(Dato fin de mes),"IPC: Bogotá, D.C.(Dato fin de mes)",IPC: Cali(Dato fin de mes),IPC: Bucaramanga(Dato fin de mes),IPC: Sin alimentos(Dato fin de mes),IPC: Regulados(Dato fin de mes),IPC: de alimentos(Dato fin de mes),Índice de Precios del Productor (IPP)(Dato fin de mes)_x,Índice de Precios del Productor (IPP)(Dato fin de mes)_y,"Índice de Precios del Productor (IPP), Producción nacional - Explotación de minas y canteras(Dato fin de mes)","Índice de Precios del Productor (IPP), Oferta interna - Total(Dato fin de mes)","Índice de Precios del Productor (IPP), Oferta interna - Agricultura, ganadería, caza, silvicultura y pesca(Dato fin de mes)","Índice de Precios del Productor (IPP), Oferta interna - Industria manufactureras(Dato fin de mes)","Índice de Precios del Productor (IPP), Según procedencia - producidos y consumidos(Dato fin de mes)","Índice de Precios del Productor (IPP), Según procedencia - Importados(Dato fin de mes)","Índice de Precios del Productor (IPP), Según procedencia - Exportados(Dato fin de mes)","Índice de Precios del Productor (IPP), Según uso o destino económico - Consumo intermedio(Dato fin de mes)","Índice de Precios del Productor (IPP), Según uso o destino económico - Consumo final(Dato fin de mes)","Índice de Precios del Productor (IPP), Según uso o destino económico - Bienes de capital(Dato fin de mes)",Tasa global de participación(Dato fin de mes),"Tasa de ocupación, según nivel educativo: sin educación superior(Dato fin de mes)","Tasa de desempleo, según nivel educativo: sin educación superior(Dato fin de mes)",Tasa de desempleo - total nacional(Dato fin de mes),Tasa de ocupación - total nacional(Dato fin de mes),Pulso Económico Regional - Comercio(Dato fin de mes),Pulso Económico Regional - Transporte(Dato fin de mes),Pulso Económico Regional - Agropecuario(Dato fin de mes),Pulso Económico Regional - Financiero(Dato fin de mes),Pulso Económico Regional - Vivienda(Dato fin de mes),Pulso Económico Regional - Nacional(Dato fin de mes),Pulso Económico Regional - Antioquia(Dato fin de mes),Pulso Económico Regional - Suroccidente(Dato fin de mes),Pulso Económico Re

In [ ]:
df.sample(3)

,fecha,producto,tasa,monto,anomes,vari_monto_m1,vari_tasa_m1,y_vari_anual_monto,vari_anual_tasa,TRM(Dato fin de mes),Euro - COP/EUR - Tasa media(Dato fin de mes),Libra esterlina - COP/GBP - Tasa media(Dato fin de mes),Índice de precios de la vivienda nueva (IPVNBR) - Índice real - Bogotá(Dato fin de mes),Índice de precios de la vivienda nueva (IPVNBR) - Índice real - Medellín(Dato fin de mes),Índice de precios de la vivienda nueva (IPVNBR) - Índice real - Cali(Dato fin de mes),Índice de precios de la vivienda nueva (IPVNBR) - Índice real - Agregado(Dato fin de mes),Inflación total anual(Dato fin de mes),Inflación sin alimentos(Dato fin de mes),Inflación sin alimentos ni regulados(Dato fin de mes),"Inflación de bienes, sin alimentos ni regulados(Dato fin de mes)","Inflación de servicios, sin alimentos ni regulados(Dato fin de mes)",Inflación de regulados(Dato fin de mes),"Inflación, Prendas de vestir y calzado(Dato fin de mes)","Inflación, Muebles, artículos para el hogar y para la conservación ordinaria del hogar(Dato fin de mes)","Inflación, Salud(Dato fin de mes)","Inflación, Información y comunicación(Dato fin de mes)","Inflación, Recreación y cultura(Dato fin de mes)","Inflación, Educación(Dato fin de mes)","Inflación, Restaurantes y hoteles(Dato fin de mes)","Inflación, Bienes y servicios diversos(Dato fin de mes)",Inflación de alimentos(Dato fin de mes),Inflación de alimentos y regulados(Dato fin de mes),Índice de Precios al Consumidor (IPC)(Dato fin de mes),IPC: Alimentos y bebidas no alcohólicas(Dato fin de mes),"IPC: Alojamiento, agua, electricidad, gas y otros combustibles(Dato fin de mes)","IPC: Muebles, artículos para el hogar y para la conservación ordinaria del hogar(Dato fin de mes)",IPC: Transporte(Dato fin de mes),IPC: Información y comunicación(Dato fin de mes),IPC: Educación(Dato fin de mes),IPC: Bienes y servicios diversos(Dato fin de mes),"IPC: Bogotá, D.C.(Dato fin de mes)",IPC: Cali(Dato fin de mes),IPC: Bucaramanga(Dato fin de mes),IPC: Sin alimentos(Dato fin de mes),IPC: Regulados(Dato fin de mes),IPC: de alimentos(Dato fin de mes),Índice de Precios del Productor (IPP)(Dato fin de mes)_x,Índice de Precios del Productor (IPP)(Dato fin de mes)_y,"Índice de Precios del Productor (IPP), Producción nacional - Explotación de minas y canteras(Dato fin de mes)","Índice de Precios del Productor (IPP), Oferta interna - Total(Dato fin de mes)","Índice de Precios del Productor (IPP), Oferta interna - Agricultura, ganadería, caza, silvicultura y pesca(Dato fin de mes)","Índice de Precios del Productor (IPP), Oferta interna - Industria manufactureras(Dato fin de mes)","Índice de Precios del Productor (IPP), Según procedencia - producidos y consumidos(Dato fin de mes)","Índice de Precios del Productor (IPP), Según procedencia - Importados(Dato fin de mes)","Índice de Precios del Productor (IPP), Según procedencia - Exportados(Dato fin de mes)","Índice de Precios del Productor (IPP), Según uso o destino económico - Consumo intermedio(Dato fin de mes)","Índice de Precios del Productor (IPP), Según uso o destino económico - Consumo final(Dato fin de mes)","Índice de Precios del Productor (IPP), Según uso o destino económico - Bienes de capital(Dato fin de mes)",Tasa global de participación(Dato fin de mes),"Tasa de ocupación, según nivel educativo: sin educación superior(Dato fin de mes)","Tasa de desempleo, según nivel educativo: sin educación superior(Dato fin de mes)",Tasa de desempleo - total nacional(Dato fin de mes),Tasa de ocupación - total nacional(Dato fin de mes),Pulso Económico Regional - Comercio(Dato fin de mes),Pulso Económico Regional - Transporte(Dato fin de mes),Pulso Económico Regional - Agropecuario(Dato fin de mes),Pulso Económico Regional - Financiero(Dato fin de mes),Pulso Económico Regional - Vivienda(Dato fin de mes),Pulso Económico Regional - Nacional(Dato fin de mes),Pulso Económico Regional - Antioquia(Dato fin de mes),Pulso Económico Regional - Suroccidente(Dato fin de mes),Pulso Económico Re

In [ ]:
var = "monto"
cols = [x for x in df if x not in ["fecha", "producto", "anomes", "_merge"]]
ax = df[df["producto"] == "Créditos Comerciales (Ordinario)"][cols]
ax = ax.iloc[:len(ax)-28]
ax

,tasa,monto,vari_monto_m1,vari_tasa_m1,y_vari_anual_monto,vari_anual_tasa,TRM(Dato fin de mes),Euro - COP/EUR - Tasa media(Dato fin de mes),Libra esterlina - COP/GBP - Tasa media(Dato fin de mes),Índice de precios de la vivienda nueva (IPVNBR) - Índice real - Bogotá(Dato fin de mes),Índice de precios de la vivienda nueva (IPVNBR) - Índice real - Medellín(Dato fin de mes),Índice de precios de la vivienda nueva (IPVNBR) - Índice real - Cali(Dato fin de mes),Índice de precios de la vivienda nueva (IPVNBR) - Índice real - Agregado(Dato fin de mes),Inflación total anual(Dato fin de mes),Inflación sin alimentos(Dato fin de mes),Inflación sin alimentos ni regulados(Dato fin de mes),"Inflación de bienes, sin alimentos ni regulados(Dato fin de mes)","Inflación de servicios, sin alimentos ni regulados(Dato fin de mes)",Inflación de regulados(Dato fin de mes),"Inflación, Prendas de vestir y calzado(Dato fin de mes)","Inflación, Muebles, artículos para el hogar y para la conservación ordinaria del hogar(Dato fin de mes)","Inflación, Salud(Dato fin de mes)","Inflación, Información y comunicación(Dato fin de mes)","Inflación, Recreación y cultura(Dato fin de mes)","Inflación, Educación(Dato fin de mes)","Inflación, Restaurantes y hoteles(Dato fin de mes)","Inflación, Bienes y servicios diversos(Dato fin de mes)",Inflación de alimentos(Dato fin de mes),Inflación de alimentos y regulados(Dato fin de mes),Índice de Precios al Consumidor (IPC)(Dato fin de mes),IPC: Alimentos y bebidas no alcohólicas(Dato fin de mes),"IPC: Alojamiento, agua, electricidad, gas y otros combustibles(Dato fin de mes)","IPC: Muebles, artículos para el hogar y para la conservación ordinaria del hogar(Dato fin de mes)",IPC: Transporte(Dato fin de mes),IPC: Información y comunicación(Dato fin de mes),IPC: Educación(Dato fin de mes),IPC: Bienes y servicios diversos(Dato fin de mes),"IPC: Bogotá, D.C.(Dato fin de mes)",IPC: Cali(Dato fin de mes),IPC: Bucaramanga(Dato fin de mes),IPC: Sin alimentos(Dato fin de mes),IPC: Regulados(Dato fin de mes),IPC: de alimentos(Dato fin de mes),Índice de Precios del Productor (IPP)(Dato fin de mes)_x,Índice de Precios del Productor (IPP)(Dato fin de mes)_y,"Índice de Precios del Productor (IPP), Producción nacional - Explotación de minas y canteras(Dato fin de mes)","Índice de Precios del Productor (IPP), Oferta interna - Total(Dato fin de mes)","Índice de Precios del Productor (IPP), Oferta interna - Agricultura, ganadería, caza, silvicultura y pesca(Dato fin de mes)","Índice de Precios del Productor (IPP), Oferta interna - Industria manufactureras(Dato fin de mes)","Índice de Precios del Productor (IPP), Según procedencia - producidos y consumidos(Dato fin de mes)","Índice de Precios del Productor (IPP), Según procedencia - Importados(Dato fin de mes)","Índice de Precios del Productor (IPP), Según procedencia - Exportados(Dato fin de mes)","Índice de Precios del Productor (IPP), Según uso o destino económico - Consumo intermedio(Dato fin de mes)","Índice de Precios del Productor (IPP), Según uso o destino económico - Consumo final(Dato fin de mes)","Índice de Precios del Productor (IPP), Según uso o destino económico - Bienes de capital(Dato fin de mes)",Tasa global de participación(Dato fin de mes),"Tasa de ocupación, según nivel educativo: sin educación superior(Dato fin de mes)","Tasa de desempleo, según nivel educativo: sin educación superior(Dato fin de mes)",Tasa de desempleo - total nacional(Dato fin de mes),Tasa de ocupación - total nacional(Dato fin de mes),Pulso Económico Regional - Comercio(Dato fin de mes),Pulso Económico Regional - Transporte(Dato fin de mes),Pulso Económico Regional - Agropecuario(Dato fin de mes),Pulso Económico Regional - Financiero(Dato fin de mes),Pulso Económico Regional - Vivienda(Dato fin de mes),Pulso Económico Regional - Nacional(Dato fin de mes),Pulso Económico Regional - Antioquia(Dato fin de mes),Pulso Económico Regional - Suroccidente(Dato fin de mes),Pulso Económico Regional - Bogotá(Dato f

In [ ]:
_cor = ax.corr().reset_index()
_cor = pd.melt(_cor, id_vars = "index")
_cor = _cor[_cor["index"] != _cor["variable"]]
_cor = _cor[_cor["variable"] == var].sort_values("value", ascending = False).drop(columns = "variable")
_cor.head(40)

,index,value
128,"Índice de Precios del Productor (IPP), Según uso o destino económico - Consumo final(Dato fin de...",0.8840
117,IPC: de alimentos(Dato fin de mes),0.8817
116,IPC: Regulados(Dato fin de mes),0.8792
124,"Índice de Precios del Productor (IPP), Según procedencia - producidos y consumidos(Dato fin de mes)",0.8786
104,Índice de Precios al Consumidor (IPC)(Dato fin de mes),0.8749
122,"Índice de Precios del Productor (IPP), Oferta interna - Agricultura, ganadería, caza, silvicultu...",0.8698
115,IPC: Sin alimentos(Dato fin de mes),0.8697
121,"Índice de Precios del Productor (IPP), Oferta interna - Total(Dato fin de mes)",0.8681
123,"Índice de Precios del Productor (IPP), Oferta interna - Industria manufactureras(Dato fin de mes)",0.8644
118,Índice de Precios del Productor (IPP)(Dato fin de mes)_x,0.8634


In [ ]:
_cor.tail(20)

,index,value
95,"Inflación, Muebles, artículos para el hogar y para la conservación ordinaria del hogar(Dato fin ...",0.1867
103,Inflación de alimentos y regulados(Dato fin de mes),0.1841
102,Inflación de alimentos(Dato fin de mes),0.1326
98,"Inflación, Recreación y cultura(Dato fin de mes)",0.1179
134,Tasa de ocupación - total nacional(Dato fin de mes),0.0102
136,Pulso Económico Regional - Transporte(Dato fin de mes),-0.0021
135,Pulso Económico Regional - Comercio(Dato fin de mes),-0.0030
80,vari_anual_tasa,-0.0138
75,tasa,-0.0298
143,Pulso Económico Regional - Bogotá(Dato fin de mes),-0.0476


## Feature Enginering

In [ ]:
# Deflactacion de la variabilidad

# Medias moviles del monto 

# Medias moviles de la variabilidad

# Rezagos de la variabilidad

# Rezagos del monto

# Rezagos y medias moviles de la tasa

# Margen de Intermediación (Spread): Diferencia entre las tasas de colocación y captación. Si el spread se comprime, la oferta crediticia tiende a volverse más restrictiva.   
# $$\text{Spread}_t = \text{Tasa Ponderada de Colocación}_t - \text{Tasa Ponderada de Captación}_t$$

# Tasa Usura (Límite Legal): Afecta directamente la colocación en las modalidades de tarjeta de crédito y consumo express.

# Profundización Financiera (Crédito / PIB)Mide la relación estructural entre el tamaño del sistema crediticio y la economía real.
# Fórmula:$$\text{Profundización}_t = \left( \frac{\text{Saldo Cartera Total}_t}{\text{PIB Nominal Trimestral o Anual}_t} \right) X 100$$

# Proporción de cartera vencida sobre cartera total,

# Crecimiento interanual del desembolso real.

# Variación mensual.

# Rezagos de 1, 3, 6 y 12 meses.

# Promedios móviles.

# Diferenciales de tasas.

# Variables de estacionalidad.

# Indicadores de condiciones financieras.

In [ ]:
df["covid"] = np.where("2020-03-01" <= df["fecha"] <= "2022-06-30", 1, 0)
df["crisis_2008"] = np.where("2008-09-01" <= df["fecha"] <= "2009-10-31", 1, 0)

NameError: name 'df' is not defined

In [ ]:
"""
Primera: incorporar información financiera adicional.

No tienes que limitarte a inflación, desempleo, ISE y TRM. Los spreads de tasas, indicadores bursátiles y condiciones de liquidez también pueden aportar información.

Segunda: construir indicadores derivados.

En vez de incluir únicamente las tasas de interés, puedes construir:
- Diferencial entre tasas activas y tasa de intervención.
- Diferencial entre tasas de captación y colocación.
- Variaciones de tasas.
- Pendiente de la curva de rendimientos.
Tercera: aprovechar los rezagos.

Los autores muestran que los valores actuales y rezagados de los indicadores financieros pueden contener información relevante.

Para tus datos mensuales, probaría rezagos de 1, 3, 6 y 12 meses, seleccionados dentro de la validación temporal.
"""